# 01. 공식 baseline 재현과 기본 측정 (exp000)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jennk2025/HAIKUTHON/blob/main/notebooks/01_baseline_repro.ipynb)

무료 Colab(T4) 또는 Kaggle에서 실행한다. GPU는 1장만 쓴다.

| # | 측정 | 목적 |
|---|---|---|
| 1 | 텍스트 토큰 길이 | 64토큰에서 잘리는 프롬프트 수와 잘리는 내용 |
| 2 | 스타터 키트 동봉 가중치 검증 | 공식 점수 0.7839 재현 여부, Hit@K, 순위 분포 |
| 3 | 사전학습 SigLIP zero-shot | 학습 효과를 분리해서 보기 위한 기준 |
| 4 | 후보 1:2 비율 검증 | Test(텍스트 8,000 : 이미지 16,000)와 비슷한 조건 |
| 5 | 공식 `train.py` 재실행 | 무료 GPU에서 학습 시간과 점수 |
| 6 | Test 규모 추론 시간 | 이미지 16,000 + query 8,000을 GPU 1장에서 처리하는 시간, VRAM |
| 7 | 문항 독립성 | query 순서를 바꿔도 결과가 같은지 |

결과는 `RESULTS_DIR`에 JSON으로 저장되고, 마지막 셀에 요약이 출력된다. **요약을 그대로 복사해서 전달하면 README 실험표와 PUSH_LOG에 정리한다.**

> T4에서 잰 시간은 L4 채점 서버와 다르다. L4 환산값은 추정으로만 쓴다.

In [14]:
# 저장소 받기 + 경로 설정 (Colab / Kaggle 공통). 노트북마다 맨 처음 한 번 실행한다.
import os, sys, subprocess
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # GPU 1장만 사용 (채점 서버와 같은 조건). torch import 전에 설정
REPO_URL = "https://github.com/jennk2025/HAIKUTHON.git"
if Path("/kaggle").exists() or Path("/content").exists():
    REPO_DIR = Path("/kaggle/working/HAIKUTHON" if Path("/kaggle").exists() else "/content/HAIKUTHON")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    else:                                    # 이미 받았으면 최신 코드로 갱신
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False)
else:                                        # 로컬: notebooks/ 의 상위 폴더 = 저장소
    REPO_DIR = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO_DIR))
print("REPO_DIR =", REPO_DIR)

REPO_DIR = /kaggle/working/HAIKUTHON


In [19]:
%pip install -q -r {REPO_DIR}/requirements-colab.txt

In [21]:
# 데이터 + 스타터 키트 준비.
# Colab: Drive에 데이터 묶음이 있으면 그걸 풀고, 없으면 HF에서 받은 뒤 Drive에 저장한다(처음 한 번).
# Kaggle: 오른쪽 Settings에서 Internet을 켜야 HF에서 받을 수 있다.
from haiku.setup_data import prepare, env_info
P = prepare(use_drive=True, save_to_drive=True)
DATA_DIR, KIT_DIR, WORK_DIR, RESULTS_DIR = P["data_dir"], P["kit_dir"], P["work_dir"], P["results_dir"]

[kaggle] 데이터 출처: 이미 준비됨
  work_dir     /kaggle/working/work
  data_dir     /kaggle/working/work/haikuthon-train
  kit_dir      /kaggle/working/work/starterkit
  results_dir  /kaggle/working/work/results


In [23]:
import json
ENV_INFO = env_info()
print(json.dumps(ENV_INFO, indent=2, ensure_ascii=False))
assert ENV_INFO.get("cuda_available"), "GPU가 잡히지 않았습니다. 런타임 > 런타임 유형 변경 > GPU(T4)"

{
  "env": "kaggle",
  "python": "3.13.16",
  "cpu_count": 2,
  "torch": "2.14.1+cu130",
  "cuda_available": true,
  "gpu": "Tesla T4",
  "gpu_mem_gb": 14.6,
  "visible_gpus": 1,
  "transformers": "5.22.2",
  "safetensors": "0.8.0",
  "sentencepiece": "0.2.3",
  "PIL": "11.3.0",
  "huggingface_hub": "1.33.0"
}


In [25]:
import importlib.util, time
import numpy as np
import torch
from haiku.data import load_rows, official_split, to_queries, to_candidates, gold_of
from haiku.metrics import metrics_from_ranks, score_predictions
from haiku.retrieval import load_siglip, evaluate_rows
from haiku.timing import timed, reset_peak_memory, peak_memory_gb, test_scale_inputs, save_result

rows = load_rows(DATA_DIR)
train_rows, valid_rows = official_split(rows, seed=42)
print(f"학습 {len(train_rows)} / 검증 {len(valid_rows)}")

def import_model_py(model_py, name):
    """제출 폴더의 model.py를 모듈로 불러온다 (load_model, predict)."""
    spec = importlib.util.spec_from_file_location(name, model_py)
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    return mod

SUMMARY = {"env": ENV_INFO}
def show(name, m):
    keys = ["n_queries", "score", "recall@1", "ndcg@5", "hit@5", "hit@20", "hit@100"]
    print(f"{name:40s} " + "  ".join(f"{k}={m[k]:.4f}" if isinstance(m.get(k), float) else f"{k}={m.get(k)}" for k in keys if k in m))
    if "rank_dist" in m: print(f"{'':40s} 순위 분포 {m['rank_dist']}")

학습 3200 / 검증 800


## 1. 텍스트 토큰 길이 (SigLIP 토크나이저, 최대 64토큰)

In [27]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(KIT_DIR / "sample_submission" / "models", local_files_only=True)
MAXLEN = 64
ids_all = [tok(r["prompt"])["input_ids"] for r in rows]
lens = np.array([len(x) for x in ids_all])
over = lens > MAXLEN
tok_stats = {
    "min": int(lens.min()), "mean": round(float(lens.mean()), 1), "p50": int(np.percentile(lens, 50)),
    "p90": int(np.percentile(lens, 90)), "max": int(lens.max()),
    "n_over_64": int(over.sum()), "ratio_over_64": round(float(over.mean()), 4),
}
print(tok_stats)

# 잘리는 부분에 어떤 정보가 있는지 (대략: 64번째 토큰 이후를 복원)
tails = [tok.decode(x[MAXLEN - 1:], skip_special_tokens=True) for x, o in zip(ids_all, over) if o]
kw = {"lighting": "light", "view": "view", "left/right": ("left", "right"), "above/below": ("above", "below"),
      "next to": "next to", "style": ("photograph", "render", "painting", "drawing", "sketch", "illustration")}
tail_kw = {k: sum(any(w in t.lower() for w in (v if isinstance(v, tuple) else (v,))) for t in tails) for k, v in kw.items()}
print("잘린 부분에 나오는 키워드 수:", tail_kw)
for t in tails[:5]: print(" ...", t)
SUMMARY["token_length"] = {**tok_stats, "truncated_tail_keywords": tail_kw}

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (74 > 64). Running this sequence through the model will result in indexing errors


{'min': 21, 'mean': 42.9, 'p50': 42, 'p90': 53, 'max': 77, 'n_over_64': 38, 'ratio_over_64': 0.0095}
잘린 부분에 나오는 키워드 수: {'lighting': 27, 'view': 13, 'left/right': 2, 'above/below': 1, 'next to': 0, 'style': 10}
 ... notebook is to the left of the loaf of bread
 ... from an overhead view with soft diffuse lighting
 ... diffuse lighting
 ... illustration viewed from the front with side lighting
 ... as an oil painting from an overhead view with side lighting


## 2. 스타터 키트 동봉 가중치 (공식 검증 점수 0.7839 재현)

In [29]:
kit = import_model_py(KIT_DIR / "sample_submission" / "model.py", "kit_model")
kit_model = kit.load_model(KIT_DIR / "sample_submission", "cuda")

# (a) 공식 predict()로 채점 — 제출 코드와 같은 경로
preds = kit.predict(kit_model, to_queries(valid_rows), to_candidates(valid_rows))
m_official = score_predictions(preds, gold_of(valid_rows))
show("동봉 가중치 / predict() / 후보 800", m_official)

# (b) 전체 순위로 Hit@K, 순위 분포 — (a)와 점수가 같아야 한다
ranks, _, _ = evaluate_rows(kit_model, valid_rows, valid_rows)
m_kit = metrics_from_ranks(ranks)
show("동봉 가중치 / 전체 순위 / 후보 800", m_kit)
print("predict() 점수와 차이:", round(m_kit["score"] - m_official["score"], 6))
SUMMARY["kit_weights_val800"] = m_kit

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

동봉 가중치 / predict() / 후보 800              n_queries=800  score=0.7839  recall@1=0.6925  ndcg@5=0.8753
동봉 가중치 / 전체 순위 / 후보 800                  n_queries=800  score=0.7839  recall@1=0.6925  ndcg@5=0.8753  hit@5=1.0000  hit@20=1.0000  hit@100=1.0000
                                         순위 분포 {'1': 554, '2-5': 246, '6-20': 0, '>20': 0}
predict() 점수와 차이: -0.0


## 3~4. zero-shot 기준선과 후보 1:2 비율 검증

- zero-shot은 학습을 하지 않으므로 Train 4,000쌍 전체를 검증에 써도 누수가 없다.
- 1:2 비율: 검증 query 800개, 후보 = 검증 이미지 800 + 학습 이미지 800(무작위). Test의 텍스트:이미지 = 1:2에 맞춘 설정이다. 학습된 모델은 추가 후보 800장을 학습 때 봤다는 점을 감안한다.

In [31]:
import random
extra = random.Random(0).sample(train_rows, len(valid_rows))
gallery_1to2 = valid_rows + extra

ranks, _, _ = evaluate_rows(kit_model, valid_rows, gallery_1to2)
SUMMARY["kit_weights_val800_gallery1600"] = metrics_from_ranks(ranks)
show("동봉 가중치 / 후보 1,600 (1:2)", SUMMARY["kit_weights_val800_gallery1600"])
del kit_model; torch.cuda.empty_cache()

zs_model = load_siglip("google/siglip-base-patch16-224", "cuda")
for name, q, g in [("zero-shot / 후보 800", valid_rows, valid_rows),
                   ("zero-shot / 후보 1,600 (1:2)", valid_rows, gallery_1to2),
                   ("zero-shot / 전체 4,000 : 4,000", rows, rows)]:
    ranks, _, _ = evaluate_rows(zs_model, q, g)
    m = metrics_from_ranks(ranks); show(name, m)
    SUMMARY[f"zeroshot::{name}"] = m
del zs_model; torch.cuda.empty_cache()

동봉 가중치 / 후보 1,600 (1:2)                  n_queries=800  score=0.6622  recall@1=0.5325  ndcg@5=0.7919  hit@5=0.9950  hit@20=1.0000  hit@100=1.0000
                                         순위 분포 {'1': 426, '2-5': 370, '6-20': 4, '>20': 0}


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

zero-shot / 후보 800                       n_queries=800  score=0.7682  recall@1=0.6700  ndcg@5=0.8664  hit@5=1.0000  hit@20=1.0000  hit@100=1.0000
                                         순위 분포 {'1': 536, '2-5': 264, '6-20': 0, '>20': 0}
zero-shot / 후보 1,600 (1:2)               n_queries=800  score=0.6421  recall@1=0.5062  ndcg@5=0.7779  hit@5=0.9938  hit@20=1.0000  hit@100=1.0000
                                         순위 분포 {'1': 405, '2-5': 390, '6-20': 5, '>20': 0}
zero-shot / 전체 4,000 : 4,000             n_queries=4000  score=0.4255  recall@1=0.2742  ndcg@5=0.5767  hit@5=0.8578  hit@20=0.9995  hit@100=1.0000
                                         순위 분포 {'1': 1097, '2-5': 2334, '6-20': 567, '>20': 2}


## 5. 공식 `train.py` 재실행

`train.py`는 자기 폴더의 `sample_submission/models`를 덮어쓰므로 복사본(`WORK_DIR/starterkit_run`)에서 실행한다. 기본 설정: 1에포크, 배치 16, lr 1e-5, 출력 헤드만 학습.

In [33]:
from haiku.setup_data import copy_starterkit
RUN_DIR = copy_starterkit(KIT_DIR, WORK_DIR / "starterkit_run")
cmd = [sys.executable, "train.py", "--data-dir", str(DATA_DIR), "--output", str(WORK_DIR / "submission_exp001.zip")]
print(" ".join(cmd))
t0 = time.perf_counter()
proc = subprocess.Popen(cmd, cwd=RUN_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
log = []
for line in proc.stdout:
    log.append(line); print(line, end="")
proc.wait()
train_sec = round(time.perf_counter() - t0, 1)
epoch_lines = [l.strip() for l in log if l.startswith("에포크")]
SUMMARY["train_py_default"] = {"cmd": " ".join(cmd[1:]), "returncode": proc.returncode,
                               "wall_sec": train_sec, "epoch_lines": epoch_lines}
print(SUMMARY["train_py_default"])

/usr/bin/python3 train.py --data-dir /kaggle/working/work/haikuthon-train --output /kaggle/working/work/submission_exp001.zip
학습 3,200쌍 / 검증 800쌍 / 장치 cuda
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.

Loading weights: 100%|██████████| 408/408 [00:00<00:00, 2178.47it/s]

학습: 100%|██████████| 200/200 [02:29<00:00,  1.34배치/s]
에포크 1 | loss 0.2668 | Recall@1 0.6925 | NDCG@5 0.8752 | 평균 0.7839

Writing model shards: 100%|██████████| 1/1 [00:48<00:00, 48.25s/it]
제출 폴더 저장: /kaggle/working/work/starterkit_run/sample_submission
제출 ZIP 저장: /kaggle/working/work/submission_exp001.zip
{'cmd': 'train.py --data-dir /kaggle/working/work/haikuthon-train --output /kaggle/working/work/submission_exp001.zip', 'returncode'

## 6. Test 규모 추론 시간 (이미지 16,000 + query 8,000, GPU 1장)

Train 이미지를 반복해서 16,000장 후보를 만든다. 같은 파일을 다시 읽으므로 OS 캐시 덕분에 실제보다 빠를 수 있다. 이미지 읽기(CPU)와 인코딩(GPU)을 따로 재서 병목을 본다.

In [34]:
from haiku.retrieval import read_image
run = import_model_py(RUN_DIR / "sample_submission" / "model.py", "run_model")
model = run.load_model(RUN_DIR / "sample_submission", "cuda")
network, processor = model

# (a) 이미지 읽기+전처리만 (CPU, 512장) → 16,000장 외삽 (추정)
sub = [r["image_path"] for r in rows[:512]]
t = {}
with timed(t, "decode_512"):
    for s in range(0, len(sub), 32):
        processor(images=[read_image(p) for p in sub[s:s+32]], return_tensors="pt")
decode_est = round(t["decode_512"] / 512 * 16000, 1)

# (b) predict() 전체를 Test 규모로 실측
queries, candidates = test_scale_inputs(rows, 16000, 8000)
reset_peak_memory()
with timed(t, "predict_16000x8000"):
    _ = run.predict(model, queries, candidates)
SUMMARY["timing_test_scale"] = {
    "gpu": ENV_INFO.get("gpu"), "cpu_count": ENV_INFO.get("cpu_count"),
    "predict_sec": t["predict_16000x8000"], "peak_vram_gb": peak_memory_gb(),
    "decode_preprocess_sec_est_16000": decode_est,
    "note": "측정 GPU 기준 실측. L4 환산 아님. 같은 이미지 반복으로 파일 캐시 효과 있음",
}
print(SUMMARY["timing_test_scale"])

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

{'gpu': 'Tesla T4', 'cpu_count': 2, 'predict_sec': 479.49, 'peak_vram_gb': 0.68, 'decode_preprocess_sec_est_16000': 504.1, 'note': '측정 GPU 기준 실측. L4 환산 아님. 같은 이미지 반복으로 파일 캐시 효과 있음'}


## 7. 문항 독립성 (query 순서를 섞어도 결과가 같은지)

In [35]:
qs = to_queries(valid_rows); cs = to_candidates(valid_rows)
a = run.predict(model, qs, cs)
b = run.predict(model, random.Random(1).sample(qs, len(qs)), cs)
diff_top1 = sum(a[k][0] != b[k][0] for k in a)
diff_top5 = sum(a[k] != b[k] for k in a)
SUMMARY["independence"] = {"n": len(a), "top1_changed": diff_top1, "top5_order_changed": diff_top5}
print(SUMMARY["independence"], "(fp16 배치 구성 차이로 아주 드물게 바뀔 수 있다)")

{'n': 800, 'top1_changed': 0, 'top5_order_changed': 0} (fp16 배치 구성 차이로 아주 드물게 바뀔 수 있다)


## 저장과 요약

In [36]:
import os
zip_path = WORK_DIR / "submission_exp001.zip"
SUMMARY["submission_zip_gb"] = round(os.path.getsize(zip_path) / 1024**3, 3) if zip_path.exists() else None
save_result(RESULTS_DIR, "exp000_baseline_repro", SUMMARY)
print("\n===== 아래 요약을 복사해서 전달 =====")
print(json.dumps(SUMMARY, ensure_ascii=False, indent=1, default=str))

저장: /kaggle/working/work/results/exp000_baseline_repro.json

===== 아래 요약을 복사해서 전달 =====
{
 "env": {
  "env": "kaggle",
  "python": "3.13.16",
  "cpu_count": 2,
  "torch": "2.14.1+cu130",
  "cuda_available": true,
  "gpu": "Tesla T4",
  "gpu_mem_gb": 14.6,
  "visible_gpus": 1,
  "transformers": "5.22.2",
  "safetensors": "0.8.0",
  "sentencepiece": "0.2.3",
  "PIL": "11.3.0",
  "huggingface_hub": "1.33.0"
 },
 "train_py_default": {
  "cmd": "train.py --data-dir /kaggle/working/work/haikuthon-train --output /kaggle/working/work/submission_exp001.zip",
  "returncode": 0,
  "wall_sec": 257.4,
  "epoch_lines": [
   "에포크 1 | loss 0.2668 | Recall@1 0.6925 | NDCG@5 0.8752 | 평균 0.7839"
  ]
 },
 "token_length": {
  "min": 21,
  "mean": 42.9,
  "p50": 42,
  "p90": 53,
  "max": 77,
  "n_over_64": 38,
  "ratio_over_64": 0.0095,
  "truncated_tail_keywords": {
   "lighting": 27,
   "view": 13,
   "left/right": 2,
   "above/below": 1,
   "next to": 0,
   "style": 10
  }
 },
 "timing_test_scale": {
  "